# Ana eğitim (ChangeMamba-MMBDA Tiny)

İnce defter: bütün mantık `src/hasar` paketinde. Doldurulacak bir şey yok; defter git etiketlerini okuyup
aşamasını kendi seçer:

- `protokol-kilidi` etiketi yoksa yalnız ön kontrol çalışır (eğitim yok).
- `protokol-kilidi` varsa önce ön kontrol (zaten geçtiyse atlanır), ardından eğitim çalışır (`last.pt` varsa
  kaldığı yerden devam eder, `DONE` varsa atlanır).
- `varyant-kilidi` de varsa eğitimden sonra tek Türkiye değerlendirmesi çalışır.

Yalnız varyant (Faz 8) için `CONFIG` ve `NAME` değiştirilir. Bağlantı koparsa yeniden "Run All" yeterlidir.
Ayrıntı: `docs/tasarim.md` Bölüm 8, `docs/colab.md`.

In [ ]:
# Hücre 1: Drive, aşama seçimi (git etiketleri), kurulum, sırlar
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

STEPS_OVERRIDE = None  # ör. 2000 (kısa prova); asıl eğitimde None
CONFIG = "configs/ana.yaml"  # varyant için "configs/varyant.yaml"
NAME = "ana"  # sonuç klasörü adı; varyant için "varyant"

drive.mount("/content/drive")
# Bellek parçalanmasını azaltır; alt süreçler (hasar komutları) de bu ayarı devralır.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO = Path("/content/deprem-hasar-haritasi")
DRIVE = Path("/content/drive/MyDrive/deprem-hasar")
DATA = Path("/content/data/bright")
CKPT = Path("/content/data/checkpoints")
OUT = DRIVE / "kosular"
PRE_DIR = OUT / "onkontrol" / NAME
PRE = PRE_DIR / "preflight.json"


def sh(cmd, check=True):
    """Run a shell command and stream its output into the cell (Colab hides plain subprocess output)."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc


if not REPO.exists():
    sh(f"git clone https://github.com/OgzhnTarhn/deprem-hasar-haritasi.git {REPO}")
sh(f"cd {REPO} && git fetch --tags --force --quiet origin")
# Önceki oturumun klonu kirli kalmış olabilir; etiketli commit'e geçmeden önce sıfırla.
sh(f"cd {REPO} && git reset --quiet --hard")
tags = subprocess.run(f"cd {REPO} && git tag -l", shell=True, capture_output=True, text=True).stdout
TAGS = {line.strip() for line in tags.splitlines() if line.strip()}
LOCKED = "protokol-kilidi" in TAGS
RUN_FINAL = "varyant-kilidi" in TAGS
REF = "varyant-kilidi" if RUN_FINAL else ("protokol-kilidi" if LOCKED else "origin/main")
sh(f"cd {REPO} && git checkout --quiet --force {REF}")
dirty = subprocess.run(f"cd {REPO} && git status --porcelain", shell=True, capture_output=True, text=True).stdout
assert not dirty.strip(), "Çalışma ağacı kirli; etiketli commit değiştirilmemeli."
COMMIT = subprocess.run(f"cd {REPO} && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
if RUN_FINAL:
    STAGE = "ön kontrol + eğitim + Türkiye final"
elif LOCKED:
    STAGE = "ön kontrol + eğitim"
else:
    STAGE = "ön kontrol yalnız"
print("=" * 72)
print("AŞAMA:", STAGE)
print("REF:", REF)
print("commit:", COMMIT)
print("=" * 72)
# Colab'ın torch ve timm sürümleri kullanılır; torch yeniden kurulmaz.
sh(f"pip install -q -e {REPO} einops")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN sırrı yok; indirme anonim yapılır.")
DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hücre 2: Ortam kapısı ve selective_scan çekirdeği (A100 ya da L4).
# Çekirdek bir kez derlenir ve Drive'a tekerlek (wheel) olarak kaydedilir; sonraki oturumlar saniyeler içinde kurar.
# Derlenemezse tasarım §5 gereği DamageFormer yedeğine geçilir ve bu açıkça yazılır.
import importlib
import shutil
import time

import torch

props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 2**30, 1), "GB")
# Colab A100 seçilse de bazen L4 (22 GB) veriyor; ikisi de bf16 destekler. Bellek yetmezse eğitim mikro partiye iner.
assert any(gpu in props.name for gpu in ("A100", "L4")), f"A100 ya da L4 gerekli, gelen: {props.name}"
free_gb = shutil.disk_usage("/content").free / 2**30
assert free_gb >= 60, f"Disk yetersiz: {free_gb:.0f} GB"
print("torch", torch.__version__, "CUDA", torch.version.cuda)

CM = Path("/content/ChangeMamba")
CM_COMMIT = "9ce9cec13f9ea14bc0ad91f071577ec9b3a97983"
KERNEL_DIR = CM / "kernels" / "selective_scan"
WHEELS = DRIVE / "wheels" / f"torch-{torch.__version__}-sm{props.major}{props.minor}"
BUILD_LOG = Path("/content/derleme.log")


def kernel_ready():
    importlib.invalidate_caches()
    try:
        import selective_scan_cuda_oflex  # noqa: F401

        return True
    except ImportError:
        return False


def patch_kernel_sources():
    """CUDA 13 (CCCL 3) uyumu, mamba ana deposundaki düzeltmeyle aynı.

    cub::LaneId ve cub::CTA_SYNC kaldırıldı; Volta (sm_70) artık derlenmiyor. Yalnız kullanılan
    oflex çekirdeği derlenir (süre üçte birine iner). Yama tekrar uygulanırsa bir şey değişmez.
    """
    setup = KERNEL_DIR / "setup.py"
    text = setup.read_text()
    text = text.replace("arch=compute_70,code=sm_70", "arch=compute_80,code=compute_80")
    text = text.replace('MODES = ["core", "ndstate", "oflex"]', 'MODES = ["oflex"]')
    setup.write_text(text)
    scan = KERNEL_DIR / "csrc" / "selective_scan" / "reverse_scan.cuh"
    text = scan.read_text()
    text = text.replace("cub::LaneId()", "(threadIdx.x & 0x1f)").replace("cub::CTA_SYNC()", "__syncthreads()")
    scan.write_text(text)


if not kernel_ready():
    wheels = sorted(WHEELS.glob("selective_scan-*.whl"))
    if wheels:
        sh(f"pip install -q --no-deps --force-reinstall {wheels[-1]}", check=False)
if not kernel_ready():
    if not CM.exists():
        sh(f"git clone --quiet https://github.com/ChenHongruixuan/ChangeMamba.git {CM}")
    sh(f"cd {CM} && git checkout --quiet {CM_COMMIT}")
    patch_kernel_sources()
    WHEELS.mkdir(parents=True, exist_ok=True)
    print("Çekirdek derleniyor (20-40 dk). Ayrıntılı kayıt:", BUILD_LOG, flush=True)
    t0 = time.time()
    rc = sh(
        f"cd {KERNEL_DIR} && MAX_JOBS=8 pip wheel --no-build-isolation --no-deps -w {WHEELS} . > {BUILD_LOG} 2>&1",
        check=False,
    )
    print(f"derleme {(time.time() - t0) / 60:.0f} dk, çıkış kodu {rc}")
    wheels = sorted(WHEELS.glob("selective_scan-*.whl"))
    if rc == 0 and wheels:
        sh(f"pip install -q --no-deps --force-reinstall {wheels[-1]}", check=False)
    else:
        sh(f"grep -nE 'error|fatal' {BUILD_LOG} | head -30; tail -15 {BUILD_LOG}", check=False)

KERNEL_OK = kernel_ready()
if KERNEL_OK:
    print("ÇEKİRDEK HAZIR: ana model ChangeMamba")
else:
    print("!!! ÇEKİRDEK YOK: eğitim DamageFormer yedeğiyle yapılır (tasarım §5). Yukarıdaki hata satırlarını saklayın.")

In [ ]:
# Hücre 3: Veri (VM diskine) ve ön eğitim ağırlığı
sh(f"cd {REPO} && hasar indir --root {DATA} --cache /content/data/cache")  # .done varsa atlanır
sh(f"cd {REPO} && hasar agirlik --out {CKPT}")  # yalnız ImageNet vssm_tiny (md5 denetimli)
PRETRAINED = CKPT / "vssm_tiny_0230_ckpt_epoch_262.pth"

In [ ]:
# Hücre 4: Yeniden üretim (kapı 1) ve ön kontrol kapıları -> preflight.json
import json

REPRO = REPO / "results" / "yeniden-uretim"
for key in ["siamcrnn", "unet", "damageformer"]:
    if not (REPRO / f"{key}_test.json").exists():
        sh(f"cd {REPO} && hasar yeniden-uret --model {key} --data {DATA} --out {REPRO} --cache {CKPT} --device cuda")
steps = f"--steps {STEPS_OVERRIDE}" if STEPS_OVERRIDE else ""
# Ön kontrol zaten geçtiyse kendini atlar; bu yüzden her "Run All"da çağrılır.
rc = sh(
    f"cd {REPO} && hasar on-kontrol --config {CONFIG} --data {DATA} --out {PRE_DIR} "
    f"--device cuda --pretrained {PRETRAINED} --repro-dir {REPRO}",
    check=False,
)
failed = []
if PRE.exists():
    pre = json.loads(PRE.read_text(encoding="utf-8"))
    for gate in pre.get("gates", []):
        verdict = "geçti" if gate.get("passed") else "KALDI"
        print(f"kapı {gate.get('gate')}  {verdict}  {gate.get('detail', '')}")
        if not gate.get("passed"):
            failed.append(gate.get("gate"))
        if gate.get("gate") == 9:
            print("kırpma eşiği (clip):", gate.get("value", {}).get("clip"))
if rc != 0:
    raise AssertionError(
        f"Ön kontrol kapılarından biri geçmedi (kapılar: {failed}); eğitim başlamaz.\n"
        f"preflight.json: {PRE}\nBu dosyayı Claude'a gönderin."
    )
if not LOCKED:
    print("Ön kontrol geçti. Eğitim `protokol-kilidi` etiketini bekliyor (Claude atacak).")
    TRAIN = False
else:
    TRAIN = True

In [ ]:
# Hücre 5: Ana eğitim (last.pt varsa tam durumla devam eder, DONE varsa atlanır)
if TRAIN:
    sh(
        f"cd {REPO} && hasar egit --config {CONFIG} --data {DATA} --out {OUT} "
        f"--device cuda --pretrained {PRETRAINED} --commit {COMMIT} --preflight {PRE} {steps}"
    )
else:
    print("Eğitim atlandı: `protokol-kilidi` etiketi yok, yalnız ön kontrol aşamasındayız.")

In [ ]:
# Hücre 6: Türkiye'de tek değerlendirme (Faz 6/8). Eğitim DONE değilse, protokol-kilidi ve varyant-kilidi
# etiketleri HEAD'in atası değilse ya da çalışma ağacı kirliyse çalışmaz. Çıktılar depo dışına (Drive) yazılır,
# böylece ağaç temiz kalır; sonra yerelde results/<NAME>/ altına kopyalanıp commit'lenir.
if RUN_FINAL and TRAIN:
    RES = DRIVE / "sonuclar" / NAME
    INST = Path("/content/data/ornekler")
    CACHE = Path("/content/data/final_cache")
    sh(f"cd {REPO} && hasar ornekler --dest {INST} --cache /content/data/cache")  # yalnız sabit 1.001 kare
    if not (RES / "sicaklik.json").exists():  # sıcaklık yalnız 13 olayın doğrulamasında seçilir
        sh(f"cd {REPO} && hasar sicaklik --config {CONFIG} --data {DATA} --runs {OUT} --device cuda --out {RES}")
    if (RES / "turkiye.json").exists():
        print("turkiye.json var: Türkiye değerlendirmesi zaten yapıldı, ikinci kez çalışmaz.")
    else:
        sh(
            f"cd {REPO} && hasar final --config {CONFIG} --data {DATA} --runs {OUT} --device cuda "
            f"--out {RES} --cache {CACHE} --instances {INST}"
        )
    sh(f"mkdir -p {DRIVE / 'final_cache'} && cp -rn {CACHE}/. {DRIVE / 'final_cache'}/")
    print("Sonuçlar:", RES)
else:
    print("Türkiye değerlendirmesi atlandı: `varyant-kilidi` etiketi yok ya da eğitim bu oturumda çalışmadı.")

In [ ]:
# Hücre 7: Kopyaları doğrula ve çalışma zamanını bırak (CU yakımı durur)
import json

from google.colab import runtime

done = list(OUT.glob("*/DONE"))
print("DONE:", [str(p) for p in done])
for p in done:
    print(json.loads(p.read_text(encoding="utf-8")))
runtime.unassign()